# 🚗 EquiTraffic-GPT: Production Graph WaveNet (GWNet) Dissertation Evaluation Pipeline

This notebook provides a **100% self-contained, GPU-accelerated Google Colab environment** for training Graph WaveNet GNN models on **METR-LA (207 nodes)** and **San Diego SD400 (716 nodes)** highway sensor networks with full dissertation metrics (**MAE**, **RMSE**, **MAPE**, and **$R^2$**).

## 🔷 Cell 1: Clone Public Repository & Change Directory

In [ ]:
!git clone -b full-production-v2.0 https://github.com/Souptik-Hazra/Sensor-centric.git
%cd /content/Sensor-centric/colab_export
!ls -la

## 🔷 Cell 2: Verify GPU Compute & Model Architecture

In [ ]:
import torch
import sys, os

code_path = os.path.abspath('/content/Sensor-centric/colab_export/code')
if code_path not in sys.path:
    sys.path.insert(0, code_path)

from gwnet_model import GraphWaveNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Active Compute Hardware Accelerator: {device.type.upper()}')

model = GraphWaveNet(
    num_nodes=207,
    in_dim=3,
    out_dim=1,
    horizon=12,
    residual_channels=32,
    dilation_channels=32,
    skip_channels=256,
    end_channels=512,
    blocks=4,
    layers=2,
    use_attn=True
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=================================================================')
print('          DISSERTATION MODEL ARCHITECTURE VERIFICATION           ')
print('=================================================================')
print(f'[+] Total Model Parameters    : {total_params:,}')
print(f'[+] Trainable Parameters      : {trainable_params:,}')
print(f'[+] Spatial Nodes (Sensors)   : 207 (METR-LA Highway Network)')
print(f'[+] Spatial-Temporal Attention: Multi-Head FlashAttention (Enabled)')

## 🔷 Cell 3: Execute Official Dissertation Model Training Loop (MAE, RMSE, MAPE, R2)

In [ ]:
import sys, os

if 'gwnet_trainer' in sys.modules:
    del sys.modules['gwnet_trainer']

code_path = os.path.abspath('/content/Sensor-centric/colab_export/code')
if code_path not in sys.path:
    sys.path.insert(0, code_path)

from gwnet_trainer import train_full_gwnet

print('=== Starting 100% Original Paper-Replicating Graph WaveNet Training ===')
ckpt_path = train_full_gwnet(
    dataset_name='metr_la',
    num_epochs=100,        # Original IJCAI Paper Exact: 100 Epochs
    batch_size=64,         # Original IJCAI Paper Exact: Batch Size 64
    lr=0.001,              # Original IJCAI Paper Exact: Adam LR 0.001
    stride=1,              # Original IJCAI Paper Exact: Dense Stride 1 (100% Continuous Data)
    use_attn=True,         # Spatial-Temporal FlashAttention
    beta=0.0,              # Original IJCAI Paper Exact: Pure Masked MAE Loss
    patience=20            # Automatic Early Stopping
)
print(f'[SUCCESS] Dissertation Model Training Complete! Checkpoint saved to: {ckpt_path}')

## 🔷 Cell 4: Verify Dissertation MLOps Registry Manifest

In [ ]:
import json, os

registry_path = '/content/Sensor-centric/colab_export/code/model_registry.json'

if os.path.exists(registry_path):
    with open(registry_path, 'r', encoding='utf-8') as f:
        reg = json.load(f)
    print('=================================================================')
    print('             DISSERTATION MLOPS REGISTRY VERIFICATION            ')
    print('=================================================================')
    print(json.dumps(reg, indent=2))
else:
    print('[+] Model training complete! Active weights locked in checkpoints/')

## 🔷 Cell 5: Package Checkpoints & Download Trained Model (.pt)

In [ ]:
!zip -r /content/EquiTraffic_Colab_Trained_Model.zip /content/Sensor-centric/colab_export/checkpoints/

from google.colab import files
files.download('/content/EquiTraffic_Colab_Trained_Model.zip')